# Lambda Sweep Analysis

Interactive helper for plotting metrics against `adv_reward_weight_drive` from failure mining CSVs.

Set `CSV_PATH`, run all cells, then use `plot_metric(...)` or the widget UI if `ipywidgets` is installed.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "pufferlib").exists() and (path / "notebooks").exists():
            return path
    return start


REPO_ROOT = find_repo_root()

# Change this to any lambda-sweep mining CSV.
CSV_PATH = REPO_ROOT / "failure_runs/all_maps_640/episodes_agents64_seed2904_target_strong_target.csv"

# Plots and summaries are written next to the CSV by default.
OUTPUT_DIR = CSV_PATH.parent / "lambda_sweep_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH, OUTPUT_DIR

In [ ]:
def load_lambda_sweep(csv_path):
    df = pd.read_csv(csv_path)
    lambda_candidates = ["adv_reward_weight_drive", "adv_drive_weight", "lambda", "drive_weight"]
    lambda_col = next((col for col in lambda_candidates if col in df.columns), None)
    if lambda_col is None:
        raise ValueError(f"No lambda column found. Tried: {lambda_candidates}")

    df = df.copy()
    df[lambda_col] = pd.to_numeric(df[lambda_col], errors="coerce")
    df = df[df[lambda_col].notna()]
    return df, lambda_col


df, LAMBDA_COL = load_lambda_sweep(CSV_PATH)
numeric_cols = sorted(df.select_dtypes(include=[np.number]).columns.tolist())
default_metrics = [
    col
    for col in [
        "did_target_fail",
        "did_target_collide",
        "did_target_offroad",
        "did_target_run_light",
        "target_episode_return",
        "target_mean_reward",
        "target_puffer_score",
        "target_progress_ratio",
        "target_ttc_within_bound_rate",
        "target_num_goals_reached",
        "target_collision_severity",
        "target_collision_responsibility",
    ]
    if col in df.columns
]

print(f"Loaded {len(df):,} rows from {CSV_PATH}")
print(f"Lambda column: {LAMBDA_COL}")
print(f"Lambda values: {sorted(df[LAMBDA_COL].dropna().unique())}")
print(f"Default metrics: {default_metrics}")
df.head()

In [ ]:
def summarize_metric(metric, by=None, df=df, lambda_col=LAMBDA_COL):
    if metric not in df.columns:
        raise ValueError(f"Unknown metric: {metric}")

    group_cols = [lambda_col] if by is None else [lambda_col, by]
    clean = df[group_cols + [metric]].copy()
    clean[metric] = pd.to_numeric(clean[metric], errors="coerce")
    clean = clean[clean[metric].notna()]

    summary = (
        clean.groupby(group_cols, dropna=False)[metric]
        .agg(n="count", mean="mean", std="std")
        .reset_index()
        .sort_values(group_cols)
    )
    summary["sem"] = summary["std"] / np.sqrt(summary["n"].clip(lower=1))
    return summary


def plot_metric(metric, by=None, errorbar="sem", save=True, show_table=True, figsize=None):
    summary = summarize_metric(metric, by=by)
    figsize = figsize or ((8, 5) if by else (6, 4))
    fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)

    if by is None:
        yerr = summary[errorbar] if errorbar in summary.columns else None
        ax.errorbar(summary[LAMBDA_COL], summary["mean"], yerr=yerr, marker="o", linewidth=2, capsize=3)
    else:
        for name, part in summary.groupby(by, dropna=False):
            yerr = part[errorbar] if errorbar in part.columns else None
            ax.errorbar(
                part[LAMBDA_COL], part["mean"], yerr=yerr, marker="o", linewidth=1.8, capsize=2, label=str(name)
            )
        ax.legend(fontsize=8, ncols=2)

    ax.set_title(f"{metric} vs {LAMBDA_COL}" + (f" by {by}" if by else ""))
    ax.set_xlabel(LAMBDA_COL)
    ax.set_ylabel(metric)
    ax.set_xticks(sorted(summary[LAMBDA_COL].dropna().unique()))
    ax.tick_params(axis="x", rotation=45)

    if save:
        suffix = f"_by_{by}" if by else ""
        path = OUTPUT_DIR / f"{metric}_vs_lambda{suffix}.png"
        fig.savefig(path, dpi=160)
        print(f"Wrote {path}")

    plt.show()
    if show_table:
        display(summary)
    return summary


def plot_metrics(metrics=default_metrics, by=None, errorbar="sem", save=True):
    summaries = {}
    for metric in metrics:
        summaries[metric] = plot_metric(metric, by=by, errorbar=errorbar, save=save, show_table=False)
    return summaries


def write_wide_summary(metrics=default_metrics):
    parts = []
    for metric in metrics:
        summary = summarize_metric(metric)
        part = summary[[LAMBDA_COL, "n", "mean", "std", "sem"]].rename(
            columns={
                "n": f"{metric}_n",
                "mean": f"{metric}_mean",
                "std": f"{metric}_std",
                "sem": f"{metric}_sem",
            }
        )
        parts.append(part)

    wide = parts[0]
    for part in parts[1:]:
        wide = wide.merge(part, on=LAMBDA_COL, how="outer")
    wide = wide.sort_values(LAMBDA_COL)
    out = OUTPUT_DIR / "lambda_sweep_wide_summary.csv"
    wide.to_csv(out, index=False)
    print(f"Wrote {out}")
    return wide

## Quick Plots

Edit the metric list or add `by="map_name"` to split curves by map.

In [ ]:
# Main curves.
plot_metrics(
    [
        "did_target_fail",
        "target_episode_return",
        "target_mean_reward",
    ],
    by=None,
)

# Optional map split.
if "map_name" in df.columns:
    plot_metric("did_target_fail", by="map_name", show_table=False)

wide_summary = write_wide_summary(default_metrics)
wide_summary.head()

## Optional Interactive UI

This cell uses `ipywidgets` if installed. If not, use `plot_metric(...)` directly.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    metric_picker = widgets.SelectMultiple(
        options=numeric_cols,
        value=tuple(default_metrics[:3]),
        description="Metrics",
        rows=12,
        layout=widgets.Layout(width="520px"),
    )
    by_options = [None] + [col for col in ["map_name", "dataset_name", "scenario_id"] if col in df.columns]
    by_picker = widgets.Dropdown(options=by_options, value=None, description="Split by")
    error_picker = widgets.Dropdown(options=["sem", "std", None], value="sem", description="Error")
    save_picker = widgets.Checkbox(value=True, description="Save PNGs")
    button = widgets.Button(description="Plot", button_style="primary")
    output = widgets.Output()

    def on_click(_):
        with output:
            clear_output(wait=True)
            selected = list(metric_picker.value)
            if not selected:
                print("Select at least one metric.")
                return
            plot_metrics(selected, by=by_picker.value, errorbar=error_picker.value, save=save_picker.value)

    button.on_click(on_click)
    display(widgets.VBox([metric_picker, widgets.HBox([by_picker, error_picker, save_picker, button]), output]))
except ImportError:
    print("ipywidgets is not installed. Use plot_metric(...) and plot_metrics(...) directly.")